# Atelier 2 (matin) : structures natives et NumPy

Notebook de référence du formateur, noyau **Python (venv-base)**.

# PARTIE 2 : LES STRUCTURES DE DONNÉES NATIVES

Cette partie et la suivante se font dans le notebook `atelier2`, avec le noyau **Python (venv-base)**.

## Cellule 1 : Vérification de l'environnement

- **Interpréteur** : `sys.executable` doit pointer vers `.venv-base`.
- **NumPy** : présent dans cet environnement ; `pandas` ne l'est pas encore.
- **Dossier actif** : doit contenir `data/` et `energie_utils.py`.

In [1]:
# === CELLULE 1 : Vérification de l'environnement ===
import sys
from pathlib import Path

import numpy as np

print(f"Python       : {sys.version.split()[0]}")
print(f"Interpréteur : {sys.executable}")
print(f"NumPy        : {np.__version__}")
print(f"Dossier      : {Path.cwd().name}")
print(f"Fichiers     : {sorted(p.name for p in Path('data').glob('*.csv'))}")
print(f"Module A1    : {Path('energie_utils.py').exists()}")

Python       : 3.12.3
Interpréteur : C:\Users\formation\formation_python_fabric\.venv-base\Scripts\python.exe
NumPy        : 2.5.3
Dossier      : formation_python_fabric
Fichiers     : ['consommation_30jours.csv', 'sites_reference.csv']
Module A1    : True


## Cellule 2 : Charger le fichier de 30 jours

- **Réutilisation** : `lire_csv` vient du module `energie_utils` (Atelier 1).
- **`list(...)`** : matérialise le générateur en **liste de dictionnaires**.
- **Volume** : 6 sites × 24 h × 30 jours, plus des doublons.

In [2]:
# === CELLULE 2 : Charger les mesures ===
from energie_utils import convertir_mesure, lire_csv, parse_timestamp, qualifier_mesure

lignes = list(lire_csv("data/consommation_30jours.csv"))
print(len(lignes), "lignes")
print(type(lignes).__name__, type(lignes[0]).__name__)
print(lignes[0])

4365 lignes
list dict
{'timestamp': '2025-01-30T03:00:00', 'site_id': 'SITE_COM_001', 'consumption_mw': '0.479', 'voltage_v': '223.3', 'frequency_hz': '50.02', 'status': 'OK'}


## Cellule 3 : Listes : indexation, tranches, tri

- **Indice négatif** : `liste[-1]` est le dernier élément.
- **Tranche** : `liste[a:b:pas]`, borne de fin **exclue**.
- **`sorted`** : renvoie une nouvelle liste ; **`.sort()`** trie sur place.

In [3]:
# === CELLULE 3 : Listes ===
conso = [convertir_mesure(l["consumption_mw"]) for l in lignes[:10]]
print(conso)
print(conso[0], conso[-1], conso[2:5], conso[::3])

valides = [c for c in conso if c is not None and c >= 0]
print(sorted(valides, reverse=True)[:3])
print(min(valides), max(valides), round(sum(valides) / len(valides), 3))

[0.479, 1.288, 1.218, 0.672, 0.288, 0.662, 0.41, 2.302, 2.069, 2.874]
0.479 2.874 [1.218, 0.672, 0.288] [0.479, 0.672, 0.41, 2.874]
[2.874, 2.302, 2.069]
0.288 2.874 1.226


## Cellule 4 : Listes : ajout, retrait et méthodes utiles

- **Ajout** : `append` (un élément), `extend` (plusieurs), `insert(i, x)`.
- **Retrait** : `pop()` (dernier), `remove(x)` (première occurrence), `del liste[i]`.
- **Recherche** : `in`, `index`, `count`.

In [4]:
# === CELLULE 4 : Méthodes de liste ===
sites_vus = []
for l in lignes:
    if l["site_id"] not in sites_vus:
        sites_vus.append(l["site_id"])
print(sites_vus)

sites_vus.extend(["SITE_TEST_999"])
sites_vus.insert(0, "SITE_ENTETE")
print(sites_vus.pop(), sites_vus.index("SITE_IND_001"), sites_vus.count("SITE_IND_001"))
sites_vus.remove("SITE_ENTETE")
print(len(sites_vus))

['SITE_COM_001', 'SITE_COM_002', 'SITE_RES_002', 'SITE_IND_002', 'SITE_IND_001', 'SITE_RES_001']
SITE_TEST_999 5 1
6


## Cellule 5 : Mutabilité : alias, copie superficielle, copie profonde

- **Alias** : `b = a` ne copie pas ; les deux noms désignent la **même** liste.
- **Copie superficielle** : `a.copy()` ou `a[:]` copie la liste, pas son contenu.
- **Copie profonde** : `copy.deepcopy` copie aussi les objets internes.

In [5]:
# === CELLULE 5 : Alias et copies ===
import copy

a = [{"site": "SITE_IND_001", "mw": 2.5}, {"site": "SITE_COM_001", "mw": 1.0}]
alias = a
superficielle = a.copy()
profonde = copy.deepcopy(a)

a[0]["mw"] = 99.0
a.append({"site": "SITE_RES_001", "mw": 0.4})
print(len(alias), len(superficielle), len(profonde))
print(alias[0]["mw"], superficielle[0]["mw"], profonde[0]["mw"])

3 2 2
99.0 99.0 2.5


## Cellule 6 : Tuples : `namedtuple` (hier) et `dataclass` (aujourd'hui)

- **Tuple** : séquence **immuable** ; sert de clé de dictionnaire.
- **Hier** : `namedtuple` donne des noms aux champs.
- **Aujourd'hui (3.10+)** : `@dataclass(frozen=True, slots=True)` : types, valeurs par défaut, méthodes.

In [6]:
# === CELLULE 6 : namedtuple et dataclass ===
from collections import namedtuple
from dataclasses import dataclass

MesureNT = namedtuple("MesureNT", ["site_id", "mw", "statut"])     # Hier

@dataclass(frozen=True, slots=True)                                 # Aujourd'hui (3.10+)
class Mesure:
    site_id: str
    mw: float | None
    statut: str = "OK"

    def est_valide(self) -> bool:
        return self.mw is not None and self.mw >= 0

nt = MesureNT("SITE_IND_001", 2.5, "OK")
dc = Mesure("SITE_IND_001", 2.5)
print(nt, "|", dc)
print(nt.mw, dc.mw, dc.est_valide())

try:
    dc.mw = 0.0
except AttributeError as erreur:
    print(f"Modification refusée : {type(erreur).__name__}")

MesureNT(site_id='SITE_IND_001', mw=2.5, statut='OK') | Mesure(site_id='SITE_IND_001', mw=2.5, statut='OK')
2.5 2.5 True
Modification refusée : FrozenInstanceError


## Cellule 7 : Dictionnaires : lecture sûre et valeurs par défaut

- **`d[cle]`** : lève `KeyError` si la clé est absente.
- **`d.get(cle, defaut)`** : renvoie une valeur de repli, sans erreur.
- **`setdefault`** : lit la clé, ou la crée avec une valeur initiale.

In [7]:
# === CELLULE 7 : Dictionnaires ===
capacites = {"SITE_IND_001": 5.0, "SITE_IND_002": 3.5, "SITE_COM_001": 2.0}

print(capacites["SITE_IND_001"], capacites.get("SITE_XXX"), capacites.get("SITE_XXX", 0.0))
try:
    capacites["SITE_XXX"]
except KeyError as erreur:
    print(f"KeyError : {erreur}")

capacites.setdefault("SITE_COM_002", 1.5)
print(list(capacites.items())[-1], len(capacites))

5.0 None 0.0
KeyError : 'SITE_XXX'
('SITE_COM_002', 1.5) 4


## Cellule 8 : Compter et regrouper : `Counter` et `defaultdict`

- **`Counter`** : compte les occurrences (`most_common`).
- **`defaultdict(list)`** : crée la liste vide à la première utilisation d'une clé.
- **Cas d'usage** : regrouper les mesures par site.

In [8]:
# === CELLULE 8 : Counter et defaultdict ===
from collections import Counter, defaultdict

par_site = Counter(l["site_id"] for l in lignes)
print(par_site.most_common(3))

groupes = defaultdict(list)
for l in lignes:
    v = convertir_mesure(l["consumption_mw"])
    if qualifier_mesure(v) == "VALIDE":
        groupes[l["site_id"]].append(v)

for site_id, valeurs in sorted(groupes.items()):
    print(f"{site_id} : {len(valeurs):>4} mesures valides, moyenne {sum(valeurs)/len(valeurs):.3f} MW")

[('SITE_COM_002', 729), ('SITE_IND_001', 729), ('SITE_RES_001', 729)]
SITE_COM_001 :  662 mesures valides, moyenne 0.848 MW
SITE_COM_002 :  677 mesures valides, moyenne 0.615 MW
SITE_IND_001 :  679 mesures valides, moyenne 2.256 MW
SITE_IND_002 :  662 mesures valides, moyenne 1.615 MW
SITE_RES_001 :  655 mesures valides, moyenne 0.387 MW
SITE_RES_002 :  677 mesures valides, moyenne 0.293 MW


## Cellule 9 : Ensembles : doublons et comparaisons

- **`set`** : éléments **uniques** et non ordonnés ; recherche très rapide.
- **Opérateurs** : `|` union, `&` intersection, `-` différence, `^` différence symétrique.
- **Usage** : comparer deux listes (référentiel et données reçues).

In [9]:
# === CELLULE 9 : Ensembles ===
referentiel = {l["site_id"] for l in lire_csv("data/sites_reference.csv")}
recus = {l["site_id"] for l in lignes}
print(len(referentiel), len(recus), referentiel == recus)
print(sorted(referentiel - {"SITE_RES_002"}), "|", sorted({"A", "B"} ^ {"B", "C"}))

jours = {parse_timestamp(l["timestamp"]).day for l in lignes}
print(len(jours), sorted(set(range(1, 32)) - jours))

uniques = {tuple(l.values()) for l in lignes}
print(len(lignes), "lignes ->", len(uniques), "uniques :", len(lignes) - len(uniques), "doublons")

6 6 True
['SITE_COM_001', 'SITE_COM_002', 'SITE_IND_001', 'SITE_IND_002', 'SITE_RES_001'] | ['A', 'C']
30 [31]
4365 lignes -> 4320 uniques : 45 doublons


## Cellule 10 : Structures imbriquées et JSON

- **Imbrication** : dictionnaire de dictionnaires de listes (site → jour → valeurs).
- **`json.dumps`** : sérialise ; `indent` et `ensure_ascii=False` pour la lisibilité.
- **`default=str`** : convertit les types non JSON (dates, `Decimal`).

In [10]:
# === CELLULE 10 : Structures imbriquées ===
import json
from datetime import datetime

arbre = defaultdict(lambda: defaultdict(list))
for l in lignes[:400]:
    v = convertir_mesure(l["consumption_mw"])
    if qualifier_mesure(v) == "VALIDE":
        jour = parse_timestamp(l["timestamp"]).date().isoformat()
        arbre[l["site_id"]][jour].append(v)

site = "SITE_IND_001"
premier_jour = sorted(arbre[site])[0]
print(site, premier_jour, arbre[site][premier_jour])

extrait = {"site": site, "genere_le": datetime(2025, 2, 1, 8, 0),
           "jour": premier_jour, "valeurs": arbre[site][premier_jour][:3]}
texte = json.dumps(extrait, indent=2, ensure_ascii=False, default=str)
print(texte)
print(json.loads(texte)["genere_le"], type(json.loads(texte)["genere_le"]).__name__)

SITE_IND_001 2025-01-01 [1.833]
{
  "site": "SITE_IND_001",
  "genere_le": "2025-02-01 08:00:00",
  "jour": "2025-01-01",
  "valeurs": [
    1.833
  ]
}
2025-02-01 08:00:00 str


## Cellule 11 : Choisir la bonne structure : coût de la recherche

- **`in` sur une liste** : parcourt les éléments un à un (lent sur de gros volumes).
- **`in` sur un `set` ou un `dict`** : accès par hachage (quasi instantané).
- **`timeit`** : mesure fiable ; en IPython, `%timeit` fait la même chose.

In [11]:
# === CELLULE 11 : Liste, ensemble, dictionnaire : coût de `in` ===
import timeit

cles_liste = [f"K{i:05d}" for i in range(20_000)]
cles_set = set(cles_liste)
cles_dict = dict.fromkeys(cles_liste, 0)
cherche = "K19999"

for nom, struct in [("liste", cles_liste), ("ensemble", cles_set), ("dictionnaire", cles_dict)]:
    duree = timeit.timeit(lambda: cherche in struct, number=2_000) / 2_000
    print(f"{nom:<13}: {duree * 1e6:10.2f} µs par recherche")

liste        :     333.88 µs par recherche
ensemble     :       0.10 µs par recherche
dictionnaire :       0.06 µs par recherche


## Cellule 12 : `pairwise`, `batched` et `nlargest`

- **`itertools.pairwise`** (3.10+) : couples consécutifs `(a, b), (b, c)…`.
- **`itertools.batched`** (**3.12+**) : lots de taille fixe ; sinon repli avec `islice`.
- **`heapq.nlargest`** : les n plus grandes valeurs sans tout trier.

In [12]:
# === CELLULE 12 : itertools et heapq ===
import heapq
from itertools import islice, pairwise

serie = groupes["SITE_COM_001"][:8]
print([round(b - a, 3) for a, b in pairwise(serie)])                    # variations consécutives

try:
    from itertools import batched                                        # 3.12+
except ImportError:
    def batched(iterable, n):                                            # repli pour 3.11
        it = iter(iterable)
        while lot := tuple(islice(it, n)):
            yield lot

lots = list(batched(range(10), 4))
print(lots)
print(heapq.nlargest(3, groupes["SITE_IND_001"]))

[0.809, -0.07, -0.726, 0.354, 0.144, 0.423, -0.188]
[(0, 1, 2, 3), (4, 5, 6, 7), (8, 9)]
[3.3, 3.3, 3.298]


# PARTIE 3 : NUMPY, LE CALCUL SUR TABLEAUX

## Cellule 13 : Du liste au tableau : `dtype`, `shape`, mémoire

- **`ndarray`** : tableau de **types homogènes**, stocké de façon contiguë.
- **`dtype` / `shape` / `ndim`** : type des éléments, dimensions, nombre d'axes.
- **`nbytes`** : mémoire réellement occupée.

In [13]:
# === CELLULE 13 : Premiers tableaux NumPy ===
import sys

valeurs = [float(v) for v in groupes["SITE_IND_001"]]
tab = np.array(valeurs)

print(tab.dtype, tab.shape, tab.ndim, tab.size)
print(f"liste : {sys.getsizeof(valeurs) + sum(sys.getsizeof(v) for v in valeurs):>7} octets")
print(f"array : {tab.nbytes:>7} octets")
print(np.array([1, 2.5, 3]).dtype, np.array([1, 2, 3]).dtype, np.array([1, "a"]).dtype)

float64 (679,) 1 679
liste :   22432 octets
array :    5432 octets
float64 int64 <U21


## Cellule 14 : Créer des tableaux : `arange`, `linspace`, `zeros`, `reshape`

- **`arange(n)`** : entiers de 0 à n-1 ; **`linspace(a, b, k)`** : k valeurs équidistantes.
- **`zeros` / `ones` / `full`** : tableaux préremplis ; `np.nan` pour « inconnu ».
- **`reshape`** : change la forme sans copier les données.

In [14]:
# === CELLULE 14 : Création et forme ===
heures = np.arange(24)
print(heures)
print(np.linspace(0, 1, 5))
print(np.zeros((2, 3)), np.full((2, 2), np.nan), sep="\n")

grille = np.arange(24).reshape(4, 6)
print(grille)
print(grille.shape, grille.T.shape, grille.reshape(-1).shape)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
[0.   0.25 0.5  0.75 1.  ]
[[0. 0. 0.]
 [0. 0. 0.]]
[[nan nan]
 [nan nan]]
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]]
(4, 6) (6, 4) (24,)


## Cellule 15 : Construire la matrice sites × heures

- **Objectif** : un tableau `(6, 720)` : 6 sites, 30 jours × 24 h.
- **Position** : (site, heure absolue) calculée depuis le timestamp ; les **doublons** s'écrasent d'eux-mêmes.
- **Non valide** : codes erreur et valeurs vides deviennent `np.nan`.

In [15]:
# === CELLULE 15 : La matrice de consommation ===
sites = sorted(referentiel)
rang = {s: i for i, s in enumerate(sites)}
debut = parse_timestamp("2025-01-01 00:00:00")

conso = np.full((len(sites), 30 * 24), np.nan)
for l in lignes:
    ts = parse_timestamp(l["timestamp"])
    heure_abs = (ts - debut).days * 24 + ts.hour
    v = convertir_mesure(l["consumption_mw"])
    if qualifier_mesure(v) in ("VALIDE", "NEGATIVE"):
        conso[rang[l["site_id"]], heure_abs] = v

print(sites)
print(conso.shape, conso.dtype, f"{conso.nbytes / 1024:.1f} Ko")
print("valeurs NaN :", int(np.isnan(conso).sum()), "sur", conso.size)

['SITE_COM_001', 'SITE_COM_002', 'SITE_IND_001', 'SITE_IND_002', 'SITE_RES_001', 'SITE_RES_002']
(6, 720) float64 33.8 Ko
valeurs NaN : 350 sur 4320


## Cellule 16 : Indexation, tranches et masques

- **`conso[i, j]`** : ligne i, colonne j ; `:` signifie « tout l'axe ».
- **Tranche** : `conso[:, :24]` = toutes les lignes, premier jour.
- **Masque booléen** : `conso[conso > 3]` filtre sans boucle.

In [16]:
# === CELLULE 16 : Indexation ===
print(conso[0, :6])
print(conso[:, 12].round(2))                     # midi du 1er jour, 6 sites
print(conso[1, 24:48].shape)                     # 2e site, 2e jour
print(conso[[0, 5], :3].round(2))                # indexation par liste (fancy)

print(conso[3][conso[3] > 3.5].round(2))         # valeurs > 3,5 MW du site 4 (capacité 3,5 MW)

[0.458   nan 0.487   nan 0.548 0.544]
[1.4  1.02 3.08 2.19 0.34  nan]
(24,)
[[0.46  nan 0.49]
 [0.25 0.22 0.23]]
[5.85 4.87 4.82 5.66 5.84]


## Cellule 17 : Vectorisation : boucle contre tableau

- **Vectorisation** : une opération s'applique à **tout** le tableau, en C.
- **Taux de charge** : consommation ÷ capacité de chaque site.
- **Comparaison** : boucle Python et opération NumPy sur les mêmes données.

In [17]:
# === CELLULE 17 : Taux de charge — boucle et vectorisation ===
cap = np.array([float(l["capacity_mw"]) for l in sorted(lire_csv("data/sites_reference.csv"),
                                                          key=lambda s: s["site_id"])])
print(cap)

def taux_boucle():
    return [[c / cap[i] for c in ligne] for i, ligne in enumerate(conso.tolist())]

def taux_numpy():
    return conso / cap[:, None]

t_boucle = timeit.timeit(taux_boucle, number=20) / 20
t_numpy = timeit.timeit(taux_numpy, number=20) / 20
print(f"boucle : {t_boucle * 1e3:7.2f} ms | numpy : {t_numpy * 1e3:7.3f} ms | x{t_boucle / t_numpy:.0f}")
print(np.allclose(np.array(taux_boucle()), taux_numpy(), equal_nan=True))

[2.  1.5 5.  3.5 0.8 0.6]
boucle :    0.74 ms | numpy :   0.050 ms | x15
True


## Cellule 18 : Broadcasting : combiner des formes différentes

- **Règle** : deux formes sont compatibles si, axe par axe, elles sont égales ou l'une vaut **1**.
- **`cap[:, None]`** : ajoute un axe → forme `(6, 1)`.
- **Erreur classique** : `(6, 720)` avec `(6,)` échoue (`720 ≠ 6`).

In [18]:
# === CELLULE 18 : Broadcasting ===
taux = conso / cap[:, None]
print(conso.shape, cap.shape, cap[:, None].shape, "->", taux.shape)

try:
    conso / cap
except ValueError as erreur:
    print(f"ValueError : {erreur}")

moyenne_site = np.nanmean(conso, axis=1, keepdims=True)
ecart = conso - moyenne_site
print(moyenne_site.shape, ecart.shape, np.nanmean(ecart, axis=1).round(6))

(6, 720) (6,) (6, 1) -> (6, 720)
ValueError : operands could not be broadcast together with shapes (6,720) (6,) 
(6, 1) (6, 720) [ 0.  0. -0.  0. -0.  0.]


## Cellule 19 : Valeurs manquantes : `NaN` et fonctions `nan*`

- **Contagion** : un `NaN` dans une somme ou une moyenne rend le **résultat `NaN`**.
- **`np.nanmean`, `np.nansum`, `np.nanmax`** : ignorent les `NaN`.
- **`np.isnan`** : test correct ; `x == np.nan` est toujours faux.

In [19]:
# === CELLULE 19 : NaN ===
site0 = conso[0]
print(np.mean(site0), np.nanmean(site0).round(3))
print(np.nan == np.nan, np.isnan(np.nan))

manquants = np.isnan(conso).sum(axis=1)
print(dict(zip(sites, manquants.tolist())))
print((np.isnan(conso).mean(axis=1) * 100).round(1))

nan 0.847
False True
{'SITE_COM_001': 66, 'SITE_COM_002': 51, 'SITE_IND_001': 50, 'SITE_IND_002': 62, 'SITE_RES_001': 73, 'SITE_RES_002': 48}
[ 9.2  7.1  6.9  8.6 10.1  6.7]


## Cellule 20 : Agrégations par axe : le profil horaire moyen

- **`axis=1`** : agrège les **colonnes** (une valeur par site) ; **`axis=0`** : les lignes.
- **`reshape(6, 30, 24)`** : sépare jours et heures.
- **`argmax`** : position du maximum (heure de pointe).

In [20]:
# === CELLULE 20 : Profil horaire moyen ===
cube = conso.reshape(6, 30, 24)
profil = np.nanmean(cube, axis=1)                 # moyenne sur les 30 jours -> (6, 24)
print(cube.shape, profil.shape)

for site, courbe in zip(sites, profil):
    print(f"{site} : pointe à {courbe.argmax():02d}h ({courbe.max():.2f} MW), creux à {courbe.argmin():02d}h ({courbe.min():.2f} MW)")

(6, 30, 24) (6, 24)
SITE_COM_001 : pointe à 14h (1.31 MW), creux à 22h (0.44 MW)
SITE_COM_002 : pointe à 12h (0.92 MW), creux à 05h (0.34 MW)
SITE_IND_001 : pointe à 19h (2.71 MW), creux à 03h (1.50 MW)
SITE_IND_002 : pointe à 17h (2.00 MW), creux à 23h (1.07 MW)
SITE_RES_001 : pointe à 21h (0.55 MW), creux à 11h (0.32 MW)
SITE_RES_002 : pointe à 07h (0.43 MW), creux à 04h (0.24 MW)


## Cellule 21 : Détecter les anomalies : `where`, `clip`, percentiles

- **`np.where(cond, a, b)`** : choix vectorisé (l'équivalent d'un `if` sur tout le tableau).
- **Pic aberrant** : consommation supérieure à la capacité du site.
- **`np.nanpercentile`** : seuil statistique en complément.

In [21]:
# === CELLULE 21 : Pics aberrants ===
depasse = conso > cap[:, None]                    # NaN > x vaut False
print("pics par site :", dict(zip(sites, depasse.sum(axis=1).tolist())))

propre = np.where(depasse, np.nan, conso)
print(np.isnan(propre).sum() - np.isnan(conso).sum(), "valeurs neutralisées")

plafonne = np.clip(conso, 0, cap[:, None])
print(np.nanmax(conso, axis=1).round(2), np.nanmax(plafonne, axis=1).round(2), sep="\n")
print(np.nanpercentile(propre, [50, 95, 99]).round(3))

pics par site : {'SITE_COM_001': 5, 'SITE_COM_002': 1, 'SITE_IND_001': 0, 'SITE_IND_002': 5, 'SITE_RES_001': 1, 'SITE_RES_002': 3}
15 valeurs neutralisées
[3.59 2.06 3.3  5.85 1.08 0.96]
[2.  1.5 3.3 3.5 0.8 0.6]
[0.633 2.892 3.224]


## Cellule 22 : Tri, `argsort` et corrélation

- **`argsort`** : positions qui trieraient le tableau ; permet de retrouver *quand*.
- **`np.corrcoef`** : matrice de corrélation (ne tolère pas les `NaN`).
- **Masque commun** : on retire les heures où un des deux sites est `NaN`.

In [22]:
# === CELLULE 22 : Top pics et corrélation ===
ind1 = propre[0]
ordre = np.argsort(np.nan_to_num(ind1, nan=-1))[::-1][:3]
print([(int(i // 24 + 1), int(i % 24), round(float(ind1[i]), 2)) for i in ordre])   # (jour, heure, MW)

ok = ~np.isnan(propre[0]) & ~np.isnan(propre[1])
print(np.corrcoef(propre[0][ok], propre[1][ok]).round(3))

ok = ~np.isnan(propre[0]) & ~np.isnan(propre[4])
print(np.corrcoef(propre[0][ok], propre[4][ok])[0, 1].round(3))

[(24, 16, 1.43), (9, 13, 1.43), (6, 10, 1.43)]
[[1.    0.981]
 [0.981 1.   ]]
-0.109


## Cellule 23 : Nombres aléatoires : `np.random.seed` (hier), `default_rng` (aujourd'hui)

- **Hier** : `np.random.seed(42)` fixe un état **global** partagé.
- **Aujourd'hui** : `rng = np.random.default_rng(42)`, un générateur **local**, recommandé.
- **Reproductibilité** : même graine, mêmes tirages.

In [23]:
# === CELLULE 23 : Générateurs aléatoires ===
np.random.seed(42)                                # Hier
ancien = np.random.normal(230, 4, 3).round(1)

rng = np.random.default_rng(42)                   # Aujourd'hui
nouveau = rng.normal(230, 4, 3).round(1)
print(ancien, nouveau)

rng = np.random.default_rng(42)
print(np.array_equal(nouveau, rng.normal(230, 4, 3).round(1)))
bruit = rng.normal(0, 0.05, size=(6, 24))
print(bruit.shape, rng.choice(sites, size=3, replace=False))

[232.  229.4 232.6] [231.2 225.8 233. ]
True
(6, 24) ['SITE_COM_001' 'SITE_COM_002' 'SITE_IND_001']


## Cellule 24 : Sauvegarder : `.npy`, `.npz`, `savetxt`

- **`np.save` / `np.load`** : format binaire natif, rapide, conserve `dtype` et forme.
- **`np.savez_compressed`** : plusieurs tableaux dans une archive compressée.
- **`np.savetxt`** : texte lisible, plus lourd.

In [24]:
# === CELLULE 24 : Sauvegarde ===
Path("out").mkdir(exist_ok=True)
np.save("out/conso_30jours.npy", conso)
np.savez_compressed("out/conso_30jours.npz", conso=conso, capacites=cap, sites=np.array(sites))
np.savetxt("out/profil_horaire.csv", profil, delimiter=";", fmt="%.3f")

for nom in ("conso_30jours.npy", "conso_30jours.npz", "profil_horaire.csv"):
    print(f"{nom:<20} {Path('out', nom).stat().st_size:>7} octets")

relu = np.load("out/conso_30jours.npy")
print(np.array_equal(relu, conso, equal_nan=True), np.load("out/conso_30jours.npz")["sites"][:2])

conso_30jours.npy      34688 octets
conso_30jours.npz      10634 octets
profil_horaire.csv       864 octets
True ['SITE_COM_001' 'SITE_COM_002']


## Cellule 25 : NumPy 1.x et NumPy 2.x : ce qui a disparu

- **NumPy 2.0** a retiré plusieurs alias anciens (du code trouvé en ligne peut échouer).
- **`np.float_`, `np.NaN`, `np.product`, `np.in1d`** : remplacés par `np.float64`, `np.nan`, `np.prod`, `np.isin`.
- **Vérifier** : tester la présence plutôt que de supposer.

In [25]:
# === CELLULE 25 : Alias supprimés dans NumPy 2 ===
print("NumPy", np.__version__)
for ancien, nouveau in [("float_", "float64"), ("NaN", "nan"), ("product", "prod"), ("in1d", "isin")]:
    print(f"np.{ancien:<8} {'présent' if hasattr(np, ancien) else 'ABSENT ':<8} -> np.{nouveau:<8} {'présent' if hasattr(np, nouveau) else 'ABSENT'}")

print(np.isin([1, 5], [1, 2, 3]), np.prod([2, 3, 4]))

NumPy 2.5.3
np.float_   ABSENT   -> np.float64  présent
np.NaN      ABSENT   -> np.nan      présent
np.product  ABSENT   -> np.prod     présent
np.in1d     ABSENT   -> np.isin     présent
[ True False] 24
